# hoopstats — run the whole pipeline on a cloud GPU (Colab)

Runs `hoopstats analyze` (detection → ball tracking → shots, make/miss → shooter, zone, team → outputs) for
each video, then prints accuracy (against the labels in the bundle) and speed.

**Upload to Google Drive first**

| Drive path | What | How to make it |
|---|---|---|
| `MyDrive/hoopstats/hoopstats_bundle.zip` | code, calibrations, labels, models (~40 MB) | on the Mac: `uv run python scripts/make_colab_bundle.py` → `data/colab/hoopstats_bundle.zip` |
| `MyDrive/hoopstats/videos/IMG_0104.MOV`, … | the videos to process | copy from `data/outdoor/` |

**Runtime:** Runtime → Change runtime type → **A100 GPU** (L4/T4 also work, slower). Then Run all.

Results go to `MyDrive/hoopstats/results/<video>/` after **each** video (events.csv, stats.json, run_log.json,
detections). If the session disconnects, run all cells again: finished videos are skipped.
When done: Runtime → Disconnect and delete runtime (stops using compute units).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc

## Parameters

In [ ]:
DRIVE_DIR    = "/content/drive/MyDrive/hoopstats"
VIDEOS       = ["IMG_0104", "IMG_0105", "IMG_0106", "IMG_0107", "IMG_0108"]   # files in DRIVE_DIR/videos/<name>.MOV
BATCH        = 16          # frames per model call; 16 suits an A100, use 8 on T4/L4
WORKERS      = 1           # parallel detection processes; 4 decodes ~3x faster on an A100 (identical results)
RENDER_RANGE = "12:32"     # seconds of annotated video for the FIRST video (None = no video; "" = whole video)
REDO         = False       # True: ignore results already on Drive and process again (e.g. to re-time)
ULTRALYTICS  = "8.4.163"   # same version as the project's uv.lock

## Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil, subprocess, sys, zipfile, json, time
from pathlib import Path

bundle = Path(DRIVE_DIR) / "hoopstats_bundle.zip"
assert bundle.exists(), f"upload the bundle first: {bundle}"
ROOT = Path("/content/hoopstats")
shutil.rmtree(ROOT, ignore_errors=True)
with zipfile.ZipFile(bundle) as z:
    z.extractall("/content")
os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS}", "lap>=0.5.12"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)   # torch: Colab's own
print(subprocess.run(["hoopstats", "--help"], capture_output=True, text=True).stdout[:300])

## Run

In [ ]:
def run(cmd):
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=ROOT)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(cmd)}")

RESULTS = Path(DRIVE_DIR) / "results"
for i, v in enumerate(VIDEOS):
    res = RESULTS / v
    if (res / "events.csv").exists() and not REDO:
        print(f"== {v}: already on Drive, skipping")
        continue
    src = Path(DRIVE_DIR) / "videos" / f"{v}.MOV"
    assert src.exists(), f"missing {src}"
    local = ROOT / "data" / "outdoor" / f"{v}.MOV"
    local.parent.mkdir(parents=True, exist_ok=True)
    t = time.time(); shutil.copy(src, local); print(f"== {v}: copied from Drive in {time.time() - t:.0f} s")
    cmd = ["hoopstats", "analyze", str(local), "--out", f"out/{v}", "--batch", str(BATCH), "--device", "cuda", "--workers", str(WORKERS)]
    if i == 0 and RENDER_RANGE is not None:
        cmd += ["--render"] if RENDER_RANGE == "" else ["--render-range", RENDER_RANGE]
    run(cmd)
    # save results + detections (small) to Drive right away
    shutil.copytree(ROOT / "out" / v, res, dirs_exist_ok=True)
    shutil.copytree(ROOT / "cache" / v / "detections", res / "detections", dirs_exist_ok=True)
    shutil.copy(ROOT / "cache" / v / "ball_track.npz", res / "ball_track.npz")
    local.unlink()
    print(f"== {v}: saved to {res}")

## Accuracy and speed

In [ ]:
# bring back results of earlier sessions so the table covers every video
for v in VIDEOS:
    res = RESULTS / v
    if res.exists() and not (ROOT / "out" / v / "events.csv").exists():
        shutil.copytree(res, ROOT / "out" / v, dirs_exist_ok=True)
run([sys.executable, "scripts/eval_outputs.py", *VIDEOS])
for v in VIDEOS:
    p = ROOT / "out" / v / "run_log.json"
    if p.exists():
        log = json.loads(p.read_text())
        det = log.get("detection") or {}
        print(f"{v}: {log['processing_s'] / 60:.1f} min for {log['video_s'] / 60:.1f} min of video; "
              f"stages {log['stage_s']}; detection {det.get('fps')} fps, ms/frame {det.get('ms_per_frame')}")

## Using the results on the Mac

Copy `MyDrive/hoopstats/results/<video>/detections/` to `~/Code/basketball/cache/<video>/detections/` and
`ball_track.npz` to `~/Code/basketball/cache/<video>/`. Then `uv run hoopstats analyze data/outdoor/<video>.MOV
--out out/<video>` runs locally in ~1–2 minutes (detection is skipped).